In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from sqlalchemy.engine import create_engine
DATABASE_URL = "sqlite:///../iaCeleApp/database/local_iaCele.db"
engine = create_engine(DATABASE_URL)

In [41]:
employees_id = 18

statement = f"""
SELECT 
    i.id_products,
    p.code,
    p.name,
    SUM(i.count_qty) AS total_qty
FROM inventory_individual i
INNER JOIN products p 
    ON i.id_products = p.id
WHERE
    i.assigned_user_id = {employees_id}
    AND i.count_type = 'individual'
GROUP BY 
    i.id_products,
    p.code,
    p.name;
"""


df1 = pd.read_sql(sql=statement, con=engine)

In [42]:
nombre = 'Gretel'

ruta_carpeta = Path.home().joinpath('Documents/Inventario/A1 2025/Archivos de conteo')

wep1 = pd.read_excel(str(ruta_carpeta)+f'/{nombre} - Eq01.xlsx')
wep1.dropna(subset=f'{nombre}', inplace=True)
wep1['Código'] = wep1['Código'].astype(str)
wep1.reset_index(drop=True, inplace=True)

a = df1.drop(columns='id_products')

b = wep1.rename(columns={
    'Código': 'code',
    'Nombre': 'name',
    f'{nombre}': 'total_qty',
    })

a.equals(b)

True

In [ ]:
ruta_carpeta = Path.home().joinpath('Documents/Inventario/A1 2025/Archivos de conteo')
num_equipo = len([p for p in ruta_carpeta.iterdir() if ".Eq" in p.name])
df = {}

for n in range(num_equipo):

    if n+1 < 10:
        equipo = f'Eq0{n+1}'
    else:
        equipo = f'Eq{n+1}'

    archivos_equipo = [p for p in ruta_carpeta.iterdir() if equipo in p.name]
    equipo_names = [p.stem[:-7] for p in archivos_equipo if ' - ' + equipo in p.stem]

    cols = ['Código', 'Nombre', equipo_names[0], equipo_names[1]]

    conteo2= pd.read_excel(str(archivos_equipo[0]))
    conteo2['Código'] = conteo2['Código'].astype(str)

    df1 = pd.read_excel(str(archivos_equipo[1]))
    df1['Código'] = df1['Código'].astype(str)
    df1[equipo_names[0]] = df1[equipo_names[0]].replace(0, np.nan)
    df1.dropna(subset=[equipo_names[0]], inplace=True)
    df1.reset_index(inplace=True)

    df2 = pd.read_excel(str(archivos_equipo[2]))
    df2['Código'] = df2['Código'].astype(str)
    df2[equipo_names[1]] = df2[equipo_names[1]].replace(0, np.nan)
    df2.dropna(subset=[equipo_names[1]], inplace=True)
    df2.reset_index(inplace=True)

    dfmerge = pd.merge(df1, df2, on='id_products', how='outer')
    dfmerge['name'] = dfmerge['name_x'].fillna(dfmerge['name_y'])
    dfmerge['iguales'] = dfmerge[equipo_names[0]] == dfmerge[equipo_names[1]]

    Eq_diferentes = dfmerge.loc[dfmerge['iguales'] == False].sort_values(by='Nombre', ascending=True)[cols]
    Eq_diferentes.reset_index(drop=True, inplace=True)

    Eq_iguales = dfmerge.loc[dfmerge['iguales'] == True].sort_values(by='Nombre', ascending=True)[cols]
    Eq_iguales.reset_index(drop=True, inplace=True)

    a = conteo2[cols].sort_values(by='Nombre', ascending=True).reset_index(drop=True)
    a[equipo_names[0]] = a[equipo_names[0]].replace(0, np.nan)
    a[equipo_names[1]] = a[equipo_names[1]].replace(0, np.nan)
    b = Eq_diferentes[cols].sort_values(by='Nombre', ascending=True).reset_index(drop=True)

    if a.equals(b):
        df[equipo] = pd.concat([Eq_iguales, conteo2], ignore_index=True)

In [86]:
wep = pd.concat(df.values(), ignore_index=True)

In [87]:
wep_dupli = wep.loc[wep['Código'].isin(wep.loc[wep['Código'].duplicated()]['Código'])].sort_values(by='Nombre', ascending=True).reset_index(drop=True)
wep_dupli

,Código,Nombre,Brenda,Gretel,Conteo2,Arturo,Bianca,Eduardo,Elsa,Adilene,...,Evelin,Mariana,Carlos G.,Carlos H.,Irma,Paty,Christian,Gladys,Leticia,Tania
0,11770,"Abrazadera Sin Fin Reforzada #16 3 3/4"" a 1 1/...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11770,"Abrazadera Sin Fin Reforzada #16 3 3/4"" a 1 1/...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12743,Aguarrás D-0102 (4Lt) Sayer *12743*,NaN,NaN,31.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.0,33.0
3,12743,Aguarrás D-0102 (4Lt) Sayer *12743*,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12993,Arnés De Cuerpo Completo Con 5 Anillos #14434 ...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,10170,Tinta De Alcohol Entona-Sayer Nogal Clásico TS...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
78,10166,Tinta De Alcohol Entona-Sayer Roble TS-6620 (2...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0
79,10166,Tinta De Alcohol Entona-Sayer Roble TS-6620 (2...,NaN,NaN,6.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,NaN,NaN
80,17316,"Varilla Roscada 1/2""-13 x 60 cm Galvanizada *1...",NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,29.0,NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
wep_dupli['dupli'] = 0.0
i_col = wep_dupli.columns.get_loc('dupli')
for i in range(len(wep_dupli)):
    prove = wep_dupli.iloc[i].drop(['Código', 'Nombre']).dropna(axis=0, how='all')
    if 'Conteo2' in prove.keys():
        wep_dupli.iloc[i, i_col] = prove['Conteo2']
    elif prove.iloc[0] == prove.iloc[1]:
        wep_dupli.iloc[i, i_col] = prove.iloc[0]
    else:
        wep_dupli.iloc[i, i_col] = pd.nan

In [89]:
for cod in wep_dupli['Código'].unique():
    prov = wep_dupli.loc[wep_dupli['Código'] == cod].dropna(axis=1, how='all')
    if prov['dupli'].count() == 2 and prov['dupli'].isin([0]).sum() != 1:
        print(prov)

  Código                                             Nombre  Julio - Gregorio  \
0  11770  Abrazadera Sin Fin Reforzada #16 3 3/4" a 1 1/...               NaN   
1  11770  Abrazadera Sin Fin Reforzada #16 3 3/4" a 1 1/...               2.0   

   Yeni  Evelin  Mariana  dupli  
0   NaN     3.0      3.0    3.0  
1   2.0     NaN      NaN    2.0  
   Código                                      Nombre  Reyna  Yadira  \
20  56311  Cartucho P/Vapor Mod 6003 3M (2Pz) *56311*    1.0     1.0   
21  56311  Cartucho P/Vapor Mod 6003 3M (2Pz) *56311*    NaN     NaN   

    Carlos G.  Carlos H.  dupli  
20        NaN        NaN    1.0  
21        1.0        1.0    1.0  
   Código                          Nombre  Conteo2  Evelin  Mariana  \
30  20238  Chapa Ph P/Mueble #53C *20238*      NaN    11.0     11.0   
31  20238  Chapa Ph P/Mueble #53C *20238*     33.0     NaN      NaN   

    Carlos G.  Carlos H.  dupli  
30        NaN        NaN   11.0  
31       33.0       21.0   33.0  
   Código          